# Ranking Prediction of the Regular Season For Each Conference

In [ ]:
from IPython.utils import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import spearmanr, kendalltau
from sklearn.neural_network import MLPRegressor


with io.capture_output() as captured:
    %run data_preprocessing.ipynb

## Data Integration

For the conference ranking prediction task, we integrate:
1. **Teams data** - core team performance metrics
3. **Player aggregated stats** - team roster quality indicators

The integration strategy:
- Start with `teams_df` as the base (one row per team-season)
- Add aggregated player statistics per team-season
- Create target variable: next year's win percentage

In [ ]:
# Sort teams by tmID and year to prepare for shifting
teams_ranking_df = teams_df.sort_values(['tmID', 'year']).copy()

# Verify teams don't change conferences
conf_changes = teams_ranking_df.groupby('tmID')['confID'].nunique()
teams_with_conf_changes = conf_changes[conf_changes > 1]
if len(teams_with_conf_changes) > 0:
    print(f"{len(teams_with_conf_changes)} teams changed conferences!")
    print(teams_with_conf_changes)

# Create target: next year's win percentage
teams_ranking_df['next_year_win_pct'] = teams_ranking_df.groupby('tmID')['win_pct'].shift(-1)

# Keep only rows where we have next year's win percentage
teams_ranking_df = teams_ranking_df.dropna(subset=['next_year_win_pct'])

print(f"Base dataset shape: {teams_ranking_df.shape}")
print(f"Seasons covered: {teams_ranking_df['year'].min()} to {teams_ranking_df['year'].max()}")

In [ ]:
# Aggregate player statistics per team-season
# This represents overall roster quality and depth

player_agg_stats = players_teams_df.groupby(['tmID', 'year']).agg({
    # Per-game averages across roster
    'ppg': 'mean',
    'apg': 'mean',
    'rpg': 'mean',
    'spg': 'mean',
    'bpg': 'mean',
    'mpg': 'mean',
    
    # Efficiency metrics
    'ts_pct': 'mean',
    'efg_pct': 'mean',
    'ast_to': 'mean',
    'sbdRpg': 'mean',
    
    # Roster depth indicators
    'playerID': 'count',  # number of players used
    'GP': 'mean'  # average games played (roster stability)
}).reset_index()

player_agg_stats = player_agg_stats.rename(columns={
    'playerID': 'roster_size',
    'GP': 'avg_player_gp',
    'ppg': 'roster_avg_ppg',
    'apg': 'roster_avg_apg',
    'rpg': 'roster_avg_rpg',
    'spg': 'roster_avg_spg',
    'bpg': 'roster_avg_bpg',
    'mpg': 'roster_avg_mpg',
    'ts_pct': 'roster_avg_ts',
    'efg_pct': 'roster_avg_efg',
    'ast_to': 'roster_avg_ast_to',
    'sbdRpg': 'roster_avg_sbdR'
})

# Merge player aggregates into main dataset
teams_ranking_df = teams_ranking_df.merge(player_agg_stats, on=['tmID', 'year'], how='left')

print(f"\nAfter player integration: {teams_ranking_df.shape}")
print(f"Missing player data: {teams_ranking_df['roster_size'].isna().sum()} rows")

The name and arena columns have no predictive value.

In [ ]:
teams_ranking_df = teams_ranking_df.drop(columns=['name', 'arena'], errors='ignore')

In [ ]:
# Final dataset summary
print("\nIntegrated Dataset Summary")
print(f"Total rows: {len(teams_ranking_df)}")
print(f"Total features: {len(teams_ranking_df.columns)}")

print(f"\nTarget variable distribution:")
display(teams_ranking_df['next_year_win_pct'].describe())

In [ ]:
print("\nSample of integrated data:")
display(teams_ranking_df.head(10))

Convert playoff column to binary indicator.

In [ ]:
teams_ranking_df['playoff'] = (teams_ranking_df['playoff'] == 'Y').astype(int)

In [ ]:
display(teams_ranking_df.head(10))

In [ ]:
display(teams_ranking_df.describe())
teams_ranking_df.shape

Remove roster_avg_ast_to due to infinity and NaN values.

In [ ]:
teams_ranking_df.drop(columns=['roster_avg_ast_to'], inplace=True)

## Prediction

In [ ]:
train_years = teams_ranking_df['year'].unique()
train_years.sort()
split_points = train_years[-2:]  # Last two years for testing

print(f"Years available: {train_years}")
print(f"Training on years: {train_years[train_years < split_points[0]]}")
print(f"Testing on years: {split_points}")

# Split data temporally
train_df = teams_ranking_df[teams_ranking_df['year'] < split_points[0]].copy()
test_df = teams_ranking_df[teams_ranking_df['year'].isin(split_points)].copy()

print(f"\nTraining set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

# Feature selection on training data
feature_cols = [col for col in train_df.columns 
                if col not in ['next_year_win_pct', 'tmID', 'confID', 'year']]
X_train = train_df[feature_cols]
y_train = train_df['next_year_win_pct']

# 1. Remove low variance features
selector = VarianceThreshold(threshold=0.01)
X_train_filtered = selector.fit_transform(X_train)
selected_features = X_train.columns[selector.get_support()].tolist()

# 2. Correlation with target
X_train_temp = train_df[selected_features]
correlations = X_train_temp.corrwith(y_train).abs().sort_values(ascending=False)
top_features = correlations.head(40).index.tolist()

# 3. Remove highly correlated features
corr_matrix = X_train_temp[top_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
final_features = [f for f in top_features if f not in to_drop]

# 4. RFE with Random Forest
X_train_final = train_df[final_features]
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rfe = RFE(estimator=rf, n_features_to_select=8, step=1) 
rfe.fit(X_train_final, y_train)


# Get selected features
selected_rfe_features = X_train_final.columns[rfe.support_].tolist()
print(f"\nFinal selected features: {len(selected_rfe_features)}")
print(selected_rfe_features)

# Apply the same feature selection to test set
X_train = train_df[selected_rfe_features]
X_test = test_df[selected_rfe_features]
y_test = test_df['next_year_win_pct']

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_lr_pred = lr_model.predict(X_test_scaled)

# Display feature coefficients
feature_importance = pd.DataFrame({
    'feature': selected_rfe_features,
    'coefficient': lr_model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print("\nFeature Coefficients:")
display(feature_importance)

In [ ]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_rf_pred = rf_model.predict(X_test)

# Display feature importances
rf_feature_importance = pd.DataFrame({
    'feature': selected_rfe_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nRandom Forest Feature Importances:")
display(rf_feature_importance)

In [ ]:
# Train SVR model
svr_model = SVR(
    kernel='rbf',
    C=1.0,
    epsilon=0.1,
    gamma='scale'
)
svr_model.fit(X_train_scaled, y_train)

# Make predictions
y_svr_pred = svr_model.predict(X_test_scaled)

print(f"\nSVR Kernel: {svr_model.kernel}")
print(f"SVR Support vectors: {svr_model.n_support_}")

In [ ]:
def evaluate_ranking_predictions(test_df, y_test, y_pred):
    """
    Evaluate model by comparing predicted rankings vs actual rankings within conferences.

    Args:
        test_df: Test dataframe with tmID, confID, year
        y_test: Actual next year win percentages
        y_pred: Predicted next year win percentages
    """
    # Create evaluation dataframe
    eval_df = test_df[['tmID', 'confID', 'year']].copy()
    eval_df['actual_win_pct'] = y_test.values
    eval_df['predicted_win_pct'] = y_pred

    # Calculate actual and predicted rankings within each conference-year
    eval_df['actual_rank'] = eval_df.groupby(['confID', 'year'])['actual_win_pct'].rank(
        ascending=False, method='min'
    )
    eval_df['predicted_rank'] = eval_df.groupby(['confID', 'year'])['predicted_win_pct'].rank(
        ascending=False, method='min'
    )

    # Calculate per-conference-year Spearman and Kendall correlations
    spearman_scores = []
    kendall_scores = []
    for (conf, year), group in eval_df.groupby(['confID', 'year']):
        if len(group) > 1:  # Need at least 2 teams for correlation
            spearman, _ = spearmanr(group['actual_rank'], group['predicted_rank'])
            kendall, _ = kendalltau(group['actual_rank'], group['predicted_rank'])
            spearman_scores.append(spearman)
            kendall_scores.append(kendall)

    # Per-conference breakdown
    conf_results = []
    for conf in eval_df['confID'].unique():
        conf_years = eval_df[eval_df['confID'] == conf].groupby('year')
        mae_list = []
        spearman_list = []
        kendall_list = []
        for year, group in conf_years:
            mae = mean_absolute_error(group['actual_rank'], group['predicted_rank'])
            if len(group) > 1:
                spearman, _ = spearmanr(group['actual_rank'], group['predicted_rank'])
                kendall, _ = kendalltau(group['actual_rank'], group['predicted_rank'])
            else:
                spearman = np.nan
                kendall = np.nan
            mae_list.append(mae)
            spearman_list.append(spearman)
            kendall_list.append(kendall)
        conf_mae_avg = np.mean(mae_list)
        conf_spearman_avg = np.nanmean(spearman_list)
        conf_kendall_avg = np.nanmean(kendall_list)
        num_teams = eval_df[eval_df['confID'] == conf].groupby('year').size().iloc[0]
        conf_results.append({
            'Conference': conf,
            'MAE': conf_mae_avg,
            'Spearman': conf_spearman_avg,
            'Kendall': conf_kendall_avg,
            'Teams': num_teams
        })
    conf_metrics = pd.DataFrame(conf_results).round(3)
    display(conf_metrics)

    return eval_df

In [ ]:
# Evaluate Linear Regression rankings
print("\nLinear Regression:")
lr_ranking_eval = evaluate_ranking_predictions(test_df, y_test, y_lr_pred)

# Evaluate Random Forest rankings
print("\nRandom Forest:")
rf_ranking_eval = evaluate_ranking_predictions(test_df, y_test, y_rf_pred)

# Evaluate SVR rankings
print("\nSupport Vector Machine:")
svr_ranking_eval = evaluate_ranking_predictions(test_df, y_test, y_svr_pred)